<div align="center">
<span style="font-size: 2.5em;">Xenon S1S2 Model Performance Readout</span>
<br/>
<span style="font-size: 1.2em; color: gray;">Comprehensive evaluation of offline and online models across signal-only and signal+background configurations</span>
</div>

## Overview

This notebook reads out Xenon S1S2 model checkpoints.

- **Model Selection**: Offline and Online trained models
- **Signal Configurations**: Signal-only vs Signal+Background (mu=150 ER/kg/day)
- **Binning**: Fixed to 10 bins (training setup)

**Metrics Evaluated**:
- Best validation loss
- Best validation accuracy

## Configuration

In [1]:
# =============================
# IMPORTS AND PATH SETTINGS
# =============================

import os
import sys
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tabulate import tabulate

# Navigate to root
ROOT_NAME = "dark-matter-sbi"
while os.path.basename(os.getcwd()) != ROOT_NAME:
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\nikla\Seafile\Studium\Master\Master Thesis\dark-matter-sbi


In [2]:
# =============================
# USER CONFIGURATION
# =============================

# SPECIFY THE HALO MODEL HERE
halo = "default"  # Options: "lmc", "shm", "shmpp", "combined"

# Model parameters
n_train = 300_000
# S1S2 checkpoints were trained with architecture-specific names
# and will be set per section below.

# Training was done with fixed binning
bins = 10

# Background settings
mu_bkg = 150  # ER events per kg per day

print(f"\n{'='*60}")
print(f"HALO MODEL: {halo.upper()}")
if halo == "combined":
    print(f"Training samples: {3*n_train:,}")
else:
    print(f"Training samples: {n_train:,}")
print(f"Bins: {bins}")
print(f"Background (ERs): {mu_bkg}")
print(f"{'='*60}")


HALO MODEL: DEFAULT
Training samples: 300,000
Bins: 10
Background (ERs): 150


## Signal-Only Models

In [3]:
# =============================
# SIGNAL-ONLY: OFFLINE MODEL
# =============================

print(f"\n{'='*80}")
print("SIGNAL-ONLY (mu_bkg = 0) | OFFLINE TRAINING")
print(f"{'='*80}\n")

signal_type = "signal_only"
training_mode = "offline"
modelname = "S1S2_signal"

offline_signal_only_result = None

from configs.config import load_model_s1s2

MODELPATH = f"models/xenon/{training_mode}/{signal_type}/{halo}/{modelname}_bins{bins}_n{n_train}_{halo}.pt"

if not os.path.exists(MODELPATH):
    print(f"  Model not found: {MODELPATH}")
else:
    try:
        model, ckpt = load_model_s1s2(
            MODELPATH,
            bins=bins,
            modelname=modelname,
            device="cpu",
            print_arch=False
        )

        offline_signal_only_result = {
            "label": "Signal-only | Offline",
            "path": MODELPATH,
            "model": model,
            "checkpoint": ckpt,
            "best_val_loss": ckpt.get("best_val_loss", np.nan),
            "best_val_acc": ckpt.get("best_val_acc", np.nan),
        }

        print("  Loaded")
        print(f"    - Best Val Loss: {offline_signal_only_result['best_val_loss']:.4f}")
        print(f"    - Best Val Acc:  {offline_signal_only_result['best_val_acc']:.4f}")
    except Exception as e:
        print(f"  Error loading model: {e}")

print(f"\nLoaded {1 if offline_signal_only_result is not None else 0} offline signal-only model")


SIGNAL-ONLY (mu_bkg = 0) | OFFLINE TRAINING

  Loaded
    - Best Val Loss: 0.4108
    - Best Val Acc:  0.7792

Loaded 1 offline signal-only model


In [4]:
# =============================
# SIGNAL-ONLY: ONLINE MODEL
# =============================

print(f"\n{'='*80}")
print("SIGNAL-ONLY (mu_bkg = 0) | ONLINE TRAINING")
print(f"{'='*80}\n")

training_mode = "online"
modelname = "S1S2_signal"
online_signal_only_result = None

MODELPATH = f"models/xenon/{training_mode}/{signal_type}/{halo}/{modelname}_bins{bins}_n{n_train}_{halo}.pt"

if not os.path.exists(MODELPATH):
    print(f"  Model not found: {MODELPATH}")
else:
    try:
        model, ckpt = load_model_s1s2(
            MODELPATH,
            bins=bins,
            modelname=modelname,
            device="cpu",
            print_arch=False
        )

        online_signal_only_result = {
            "label": "Signal-only | Online",
            "path": MODELPATH,
            "model": model,
            "checkpoint": ckpt,
            "best_val_loss": ckpt.get("best_val_loss", np.nan),
            "best_val_acc": ckpt.get("best_val_acc", np.nan),
        }

        print("  Loaded")
        print(f"    - Best Val Loss: {online_signal_only_result['best_val_loss']:.4f}")
        print(f"    - Best Val Acc:  {online_signal_only_result['best_val_acc']:.4f}")
    except Exception as e:
        print(f"  Error loading model: {e}")

print(f"\nLoaded {1 if online_signal_only_result is not None else 0} online signal-only model")


SIGNAL-ONLY (mu_bkg = 0) | ONLINE TRAINING

  Loaded
    - Best Val Loss: 0.4119
    - Best Val Acc:  0.7779

Loaded 1 online signal-only model


## Signal + Background Models

In [5]:
# =============================
# SIGNAL+BACKGROUND: OFFLINE MODEL
# =============================

print(f"\n{'='*80}")
print(f"SIGNAL + BACKGROUND (mu_bkg = {mu_bkg}) | OFFLINE TRAINING")
print(f"{'='*80}\n")

signal_type = f"signal_bkg_mu{mu_bkg:.0f}"
training_mode = "offline"
modelname = "S1S2_signal_bg"

offline_signal_bkg_result = None

MODELPATH = f"models/xenon/{training_mode}/{signal_type}/{halo}/{modelname}_bins{bins}_n{n_train}_{halo}.pt"

if not os.path.exists(MODELPATH):
    print(f"  Model not found: {MODELPATH}")
else:
    try:
        model, ckpt = load_model_s1s2(
            MODELPATH,
            bins=bins,
            modelname=modelname,
            device="cpu",
            print_arch=False
        )

        offline_signal_bkg_result = {
            "label": f"Signal+Background (mu={mu_bkg}) | Offline",
            "path": MODELPATH,
            "model": model,
            "checkpoint": ckpt,
            "best_val_loss": ckpt.get("best_val_loss", np.nan),
            "best_val_acc": ckpt.get("best_val_acc", np.nan),
        }

        print("  Loaded")
        print(f"    - Best Val Loss: {offline_signal_bkg_result['best_val_loss']:.4f}")
        print(f"    - Best Val Acc:  {offline_signal_bkg_result['best_val_acc']:.4f}")
    except Exception as e:
        print(f"  Error loading model: {e}")

print(f"\nLoaded {1 if offline_signal_bkg_result is not None else 0} offline signal+background model")


SIGNAL + BACKGROUND (mu_bkg = 150) | OFFLINE TRAINING

  Loaded
    - Best Val Loss: 0.5850
    - Best Val Acc:  0.6306

Loaded 1 offline signal+background model


In [6]:
# =============================
# SIGNAL+BACKGROUND: ONLINE MODEL
# =============================

print(f"\n{'='*80}")
print(f"SIGNAL + BACKGROUND (mu_bkg = {mu_bkg}) | ONLINE TRAINING")
print(f"{'='*80}\n")

training_mode = "online"
modelname = "S1S2_signal_bg"
online_signal_bkg_result = None

MODELPATH = f"models/xenon/{training_mode}/{signal_type}/{halo}/{modelname}_bins{bins}_n{n_train}_{halo}.pt"

if not os.path.exists(MODELPATH):
    print(f"  Model not found: {MODELPATH}")
else:
    try:
        model, ckpt = load_model_s1s2(
            MODELPATH,
            bins=bins,
            modelname=modelname,
            device="cpu",
            print_arch=False
        )

        online_signal_bkg_result = {
            "label": f"Signal+Background (mu={mu_bkg}) | Online",
            "path": MODELPATH,
            "model": model,
            "checkpoint": ckpt,
            "best_val_loss": ckpt.get("best_val_loss", np.nan),
            "best_val_acc": ckpt.get("best_val_acc", np.nan),
        }

        print("  Loaded")
        print(f"    - Best Val Loss: {online_signal_bkg_result['best_val_loss']:.4f}")
        print(f"    - Best Val Acc:  {online_signal_bkg_result['best_val_acc']:.4f}")
    except Exception as e:
        print(f"  Error loading model: {e}")

print(f"\nLoaded {1 if online_signal_bkg_result is not None else 0} online signal+background model")


SIGNAL + BACKGROUND (mu_bkg = 150) | ONLINE TRAINING

  Loaded
    - Best Val Loss: 0.5848
    - Best Val Acc:  0.6308

Loaded 1 online signal+background model


## Performance Summary

In [7]:
# =============================
# SUMMARY TABLE
# =============================

results = [
    offline_signal_only_result,
    online_signal_only_result,
    offline_signal_bkg_result,
    online_signal_bkg_result,
]

rows = []
for r in results:
    if r is None:
        continue
    rows.append([
        r["label"],
        f"{r['best_val_loss']:.4f}",
        f"{r['best_val_acc']:.4f}",
    ])

print(f"\nMODEL PERFORMANCE SUMMARY (HALO={halo.upper()}, BINS={bins})")
print("="*72)
if rows:
    print(tabulate(
        rows,
        headers=["Model", "Best Val Loss", "Best Val Acc"],
        tablefmt="grid"
    ))
else:
    print("No models loaded.")


MODEL PERFORMANCE SUMMARY (HALO=DEFAULT, BINS=10)
+--------------------------------------+-----------------+----------------+
| Model                                |   Best Val Loss |   Best Val Acc |
+======================================+=================+================+
| Signal-only | Offline                |          0.4108 |         0.7792 |
+--------------------------------------+-----------------+----------------+
| Signal-only | Online                 |          0.4119 |         0.7779 |
+--------------------------------------+-----------------+----------------+
| Signal+Background (mu=150) | Offline |          0.585  |         0.6306 |
+--------------------------------------+-----------------+----------------+
| Signal+Background (mu=150) | Online  |          0.5848 |         0.6308 |
+--------------------------------------+-----------------+----------------+


## Detailed Model Information

In [8]:
# =============================
# ACCESS MODEL OBJECTS
# =============================

print("\nModel objects are stored in the following variables:")
print("\nSignal-Only Models:")
print("  - offline_signal_only_result")
print("  - online_signal_only_result")
print("\nSignal+Background Models:")
print("  - offline_signal_bkg_result")
print("  - online_signal_bkg_result")
print("\nEach entry contains:")
print("  - 'path': Model file path")
print("  - 'model': PyTorch model object")
print("  - 'checkpoint': Training checkpoint dict")
print("  - 'best_val_loss': Best validation loss")
print("  - 'best_val_acc': Best validation accuracy")
print("\nExample usage:")
print("  model = offline_signal_only_result['model']")
print("  checkpoint = offline_signal_only_result['checkpoint']")


Model objects are stored in the following variables:

Signal-Only Models:
  - offline_signal_only_result
  - online_signal_only_result

Signal+Background Models:
  - offline_signal_bkg_result
  - online_signal_bkg_result

Each entry contains:
  - 'path': Model file path
  - 'model': PyTorch model object
  - 'checkpoint': Training checkpoint dict
  - 'best_val_loss': Best validation loss
  - 'best_val_acc': Best validation accuracy

Example usage:
  model = offline_signal_only_result['model']
  checkpoint = offline_signal_only_result['checkpoint']


Summary: Only use offline! :)